# LS-NEQ 03_nonequilibrium — Nicht-Gleichgewichts-Diagnostiken

**Dossier-Stufe D** (`lsne.experiments.STAGES["D"]`, siehe `docs/DESIGN.md` §8).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Die Gitterdiagnostiken D1–D9 aus `docs/DESIGN.md` §4.2: Störungsausbreitung (Damage-Maps, Wachstumsexponent λ als präregistrierter primärer Endpunkt), Logit-Lens-Nichtmonotonie, Hysterese unter Dosisrampe mit Neustart-Null, Relaxation nach Dosissprung (τ(ε)), Wechselstatistik, Suszeptibilität, Fluktuation–Antwort und die CA-Komplexität des Symbolfelds N/E/D. Phasendiagramme über (ε, K) pro Richtung.

**Eingaben.** Stimulusbank, Richtungsbank (Stufe B auf Drive oder neu), Generierungsbudget `cfg.gen_n_new` × `cfg.gen_n_samples`.

**Ausgaben.** Tabellen `damage`, `damage_profile`, `damage_phase`, `lens`, `lens_summary`, `update_norms`, `ca`, `ca_layers`, `hysteresis`, `hysteresis_curves`, `relaxation`, `relaxation_curves`, `switch_stats`, `susceptibility`, `fdt`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/D/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/03_nonequilibrium/`; am Ende eine kurze `summary_03_nonequilibrium_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Modell → Stufe D → Damage-Maps K=0/8/28 → λ(ε, K)-Phasendiagramm → Lens-Nichtmonotonie → Hysterese → Relaxation τ(ε) → CA-Diagramme → Suszeptibilität/FDT → Zusammenfassung.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
import os

try:
    from google.colab import drive            # type: ignore
    drive.mount("/content/drive")
except Exception as e:                        # not on Colab, or the mount was refused
    print("Kein Colab-Drive:", repr(e).split("(")[0], "-> lokale Ausgabe")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "03_nonequilibrium"
STAGE = "D"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(status="reused", run_dir=str(rd.path), seconds=0.0, error="")
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(status="ok", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="")
        print(f"Stufe {stage} ok in {report[stage]['seconds']:.0f} s: {', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Stufe D ausführen (längste Stufe: Generierung D3-D7) ------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage D --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT> [--gen-n-new 64 --gen-n-samples 8]
res_D = run_stage("D")
print({k: len(v) for k, v in res_D.items()})


In [ ]:
# --- Kernvarianten (K=0, K=8/C1, K=28/seg der ersten Formulierung) und Richtungsbank ----
with guarded("Kernvarianten und Richtungsbank"):
    variants, _checks = C.get_variants(cfg, mb, ctx)
    original = C.get_original(cfg, mb, ctx, limits)
    site = C.dossier_site(mb)
    bank = C.get_bank(cfg, mb, variants, site, ctx, limits, original=original)
    std = [v for v in variants if not C.is_original(v)]
    fam0, form0 = (std[0].family_id, std[0].formulation) if std else (None, None)
    picks = []
    for (K, Cc) in C.CORE_CONDITIONS:
        m = [v for v in std if v.family_id == fam0 and v.formulation == form0 and (int(v.K), str(v.C)) == (K, Cc)]
        if m:
            picks.append((K, Cc, m[0]))
    DIR_MAP = "lang_dom" if "lang_dom" in bank else list(bank.keys())[0]
    print("Kernvarianten:", [v.tag for _, _, v in picks], "| Richtung:", DIR_MAP, "| Bankquelle:", bank.meta.get("source"))


In [ ]:
# --- D1: Damage-Maps (Layer × Position) für K=0 vs K=8 vs K=28 ------------------------
# Injection at the depth site (block input n_layers//3) at the last token of unit 9,
# D(l, t) = ||h' - h|| / ||h||; the dashed line marks the shared English tail.
with guarded("Damage-Maps"):
    from lsne.dynamics import damage_spreading, growth_exponent

    EPS_MAP = float(max(limits.damage_doses))
    fig, axes = plt.subplots(1, len(picks), figsize=(5.2 * len(picks), 4.2), squeeze=False)
    im = None
    for ax, (K, Cc, v) in zip(axes[0], picks):
        pos = C.unit_last_position(v)
        dm = damage_spreading(mb, C.ids_tensor(mb, v), C.depth_site(mb), pos, bank[DIR_MAP].vec, EPS_MAP,
                              target_id=int(v.target_id))
        lam = growth_exponent(dm, [p for p in C.tail_positions(v) if p >= pos] or [v.n_tokens - 1])["lam"]
        im = ax.imshow(np.log10(np.clip(dm.D, 1e-6, None)), aspect="auto", origin="lower", cmap="magma", vmin=-6, vmax=0,
                       interpolation="nearest")
        ax.axvline(v.tail_start - 0.5, color="cyan", lw=0.9, ls="--")
        ax.plot([pos], [dm.inject_layer], marker="^", color="white", ms=8)
        ax.set_xlim(max(-0.5, pos - 24.5), v.n_tokens - 0.5)      # sites before t0 are unperturbed (causality)
        ax.set_title(f"K={K} {Cc} ({v.tag})\nInjektion @ t={pos}, λ={lam:+.3f}", fontsize=9)
        ax.set_xlabel("Position t (gestrichelt: Beginn des englischen Schlussabschnitts)")
    axes[0, 0].set_ylabel("Snapshot ℓ (Block-Eingänge; oberster = Ausgang)")
    fig.suptitle(f"Störungsausbreitung D(ℓ, t): Richtung {DIR_MAP}, ε={EPS_MAP:g}, Injektion {C.depth_site(mb).key}", fontsize=10)
    if im is not None:
        fig.colorbar(im, ax=axes[0].tolist(), label="log10 D(ℓ, t)", shrink=0.9)
    savefig("damage_maps")
    plt.show()


In [ ]:
# --- D1: Phasendiagramm λ(ε, K) pro Richtung (Injektion 'depth', Mittel über C) ---------
with guarded("Phasendiagramm λ(ε, K)"):
    ph = table("D", "damage_phase")
    ph = ph[(ph["inject"] == "depth") & (ph["kind"] == "standard")]
    fams = [f for f in ["lang_dom", "ent", "rand", "sae"] if f in set(ph["dir_family"])]
    vmax = float(np.nanmax(np.abs(ph["lam_mean"]))) if len(ph) else 1.0
    vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1.0
    fig, axes = plt.subplots(1, len(fams), figsize=(4.6 * len(fams), 3.9), squeeze=False)
    im = None
    for ax, fam in zip(axes[0], fams):
        piv = ph[ph["dir_family"] == fam].groupby(["K", "eps"])["lam_mean"].mean().unstack("eps")
        im = ax.imshow(piv.to_numpy(), aspect="auto", origin="lower", cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.set_xticks(range(piv.shape[1]))
        ax.set_xticklabels([f"{e:g}" for e in piv.columns])
        ax.set_yticks(range(piv.shape[0]))
        ax.set_yticklabels([str(k) for k in piv.index])
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                ax.text(j, i, f"{piv.iat[i, j]:+.2f}", ha="center", va="center", fontsize=8)
        ax.set_title(f"λ — Richtung {fam}", fontsize=10)
        ax.set_xlabel("ε")
        ax.set_ylabel("K")
    if im is not None:
        fig.colorbar(im, ax=axes[0].tolist(), label="λ = Steigung von ln D über ℓ (λ > 0: Wachstum)", shrink=0.9)
    savefig("lambda_phase_eps_K")
    plt.show()
    # structure level at fixed K: lambda over (eps, C)
    k_max = int(ph[ph["C"] != "seg"]["K"].max()) if len(ph[ph["C"] != "seg"]) else None
    if k_max:
        sub = ph[(ph["K"] == k_max) & (ph["dir_family"] == fams[0])]
        print(f"λ(ε, C) bei K={k_max}, Richtung {fams[0]}:")
        show(sub.pivot_table(index="C", columns="eps", values="lam_mean"))
    print("Primärer Endpunkt (Präregistrierung): Δλ = λ(K>0, lang_dom) − λ(K=0, lang_dom) vs. derselbe Kontrast mit rand;")
    print("das Familien-Bootstrap-Intervall wird aus der Tabelle `damage` (eine Zeile pro Variante) gebildet:")
    dmg = table("D", "damage")
    if len(dmg):
        from lsne.stats import cluster_bootstrap
        d = dmg[(dmg["inject"] == "depth") & (dmg["kind"] == "standard")]
        e_max = float(d["eps"].max())
        rows = []
        for fam in [f for f in ["lang_dom", "rand"] if f in set(d["dir_family"])]:
            dd = d[(d["dir_family"] == fam) & (d["eps"] == e_max)]
            ref = dd[dd["K"] == 0].groupby("family")["lam"].mean()
            for K, g in dd[dd["K"] > 0].groupby("K"):
                diff = g.groupby("family")["lam"].mean()
                both = pd.DataFrame(dict(dlam=diff - ref.reindex(diff.index))).dropna().reset_index()
                r = cluster_bootstrap(both, "dlam", "family", n_boot=limits.n_boot, seed=SEED)
                rows.append(dict(dir_family=fam, K=int(K), eps=e_max, dlam=r["mean"], ci_low=r["ci_low"], ci_high=r["ci_high"], n_families=r["n_clusters"]))
        show(pd.DataFrame(rows))


In [ ]:
# --- D2: Logit-Lens — KL zur Endvorhersage über die Tiefe und Nichtmonotonie-Index ------
with guarded("Logit-Lens Nichtmonotonie"):
    lens = table("D", "lens")
    ls = table("D", "lens_summary")
    lens = lens[lens["C"] != "orig"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4))
    for K, g in sorted(lens.groupby("K")):
        m = g.groupby("layer")["kl_last"].mean()
        a1.plot(m.index, m.values, marker="o", ms=3, label=f"K={K}")
    a1.set_yscale("log")
    a1.set_xlabel("Snapshot ℓ")
    a1.set_ylabel("KL(p_final ‖ p_ℓ) [nat], letzte Position")
    a1.set_title("Konvergenz der Zwischenschicht-Vorhersage")
    a1.legend(fontsize=8)
    lss = ls[ls["kind"] == "standard"] if "kind" in ls else ls
    rng = np.random.default_rng(SEED)
    for i, (K, g) in enumerate(sorted(lss.groupby("K"))):
        a2.scatter(np.full(len(g), float(K)) + rng.uniform(-0.3, 0.3, len(g)), g["nmi_last"], s=14, alpha=0.6, color=f"C{i}")
        a2.plot([K - 0.4, K + 0.4], [g["nmi_last"].mean()] * 2, color=f"C{i}", lw=2)
    a2.set_xlabel("K")
    a2.set_ylabel("Nichtmonotonie-Index (letzte Position)")
    a2.set_title("Σ positiver KL-Zuwächse / Gesamtvariation (0 = monoton)")
    savefig("lens_nonmonotonicity")
    plt.show()
    if len(lss):
        show(lss.groupby(["K", "C"])[["nmi_last", "nmi_tail", "kl_last_max", "layer_of_max_kl_last", "first_layer_target_top10"]].mean())


In [ ]:
# --- D3: Hysterese — Aufwärts- vs. Abwärtsast der Dosisrampe mit Neustart-Null ---------
with guarded("Hysterese"):
    hc = table("D", "hysteresis_curves")
    hy = table("D", "hysteresis")
    DIR_GEN = "lang_dom" if "lang_dom" in set(hc["direction"]) else str(hc["direction"].iloc[0])
    sub = hc[hc["direction"] == DIR_GEN]
    tags = list(dict.fromkeys(sub["tag"]))[:6]
    fig, axes = plt.subplots(1, len(tags), figsize=(4 * len(tags), 3.8), sharey=True, squeeze=False)
    for ax, tag in zip(axes[0], tags):
        g = sub[sub["tag"] == tag]
        for br, st, lab in (("up", "o-", "aufwärts 0→ε_max"), ("down", "s-", "abwärts ε_max→0"),
                            ("restart_a", "^:", "Neustart a"), ("restart_b", "v:", "Neustart b")):
            b = g[g["branch"] == br].sort_values("level")
            ax.plot(b["eps"], b["fD"], st, ms=4, label=lab)
        a = hy[(hy["tag"] == tag) & (hy["direction"] == DIR_GEN)]
        if len(a):
            ax.set_title(f"{tag}\nA_hyst={a['area'].iloc[0]:+.3f}  Null={a['null_area'].iloc[0]:+.3f}", fontsize=9)
        ax.set_xlabel(f"Dosis ε ({DIR_GEN})")
    axes[0, 0].set_ylabel("f_D: Anteil deutscher Tokens pro Plateau")
    axes[0, 0].legend(fontsize=7)
    savefig("hysteresis_curves")
    plt.show()
    if len(hy):
        show(hy[["tag", "K", "C", "direction", "area", "null_area", "area_minus_null", "up_mean", "down_mean", "n_switches_total"]])


In [ ]:
# --- D4: Relaxation nach Dosissprung — τ(ε) und f_D(t) ---------------------------------
with guarded("Relaxation τ(ε)"):
    rx = table("D", "relaxation")
    rc = table("D", "relaxation_curves")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4))
    for (tag, d), g in rx.groupby(["tag", "direction"]):
        g = g.sort_values("eps")
        yerr = [(g["tau"] - g["tau_lo"]).clip(lower=0).fillna(0), (g["tau_hi"] - g["tau"]).clip(lower=0).fillna(0)]
        a1.errorbar(g["eps"], g["tau"], yerr=yerr, marker="o", capsize=2, ls="-" if d == "lang_dom" else "--", label=f"{tag} · {d}")
    a1.set_xlabel("ε (Quench bei t_on)")
    a1.set_ylabel("τ [Tokens] (Exponentialfit, Bootstrap-CI)")
    a1.set_yscale("log")
    a1.set_title("Kritische Verlangsamung? τ sollte im Gleichgewicht dosisunabhängig sein")
    a1.legend(fontsize=6)
    DIR_GEN = "lang_dom" if "lang_dom" in set(rc["direction"]) else str(rc["direction"].iloc[0])
    for (tag, eps), g in rc[rc["direction"] == DIR_GEN].groupby(["tag", "eps"]):
        a2.plot(g["t"], g["fD_t"], label=f"{tag} ε={eps:g}")
    a2.axvline(limits.gen_t_on, color="k", ls="--", lw=0.8)
    a2.set_xlabel("Generierungsschritt t")
    a2.set_ylabel("f_D(t) (Fenster 8)")
    a2.set_title(f"Antwort auf den Dosissprung ({DIR_GEN})")
    a2.legend(fontsize=6)
    savefig("relaxation_tau")
    plt.show()


In [ ]:
# --- D9: CA-Diagramme — Symbolfeld S(ℓ, t) ∈ {N, E, D} aus der Logit-Lens-Sprachmasse ----
with guarded("CA-Diagramme"):
    from matplotlib.colors import ListedColormap
    from lsne.dynamics import ca_symbolize
    from lsne.model import run_with_cache

    lex = C.marker_lexicon(mb.tokenizer)
    cmap = ListedColormap(["#d9d9d9", "#1f77b4", "#d62728"])
    fig, axes = plt.subplots(len(picks), 1, figsize=(13, 2.4 * len(picks)), squeeze=False)
    for ax, (K, Cc, v) in zip(axes[:, 0], picks):
        cache = run_with_cache(mb, C.ids_tensor(mb, v), "all_block_inputs")
        S = ca_symbolize(mb, cache, lex, lang_axis=bank.get("lang_dom"), thresholds=(limits.ca_threshold,))
        ax.imshow(S, aspect="auto", origin="lower", cmap=cmap, vmin=0, vmax=2, interpolation="nearest")
        # input language of every token as a strip above the field
        lang = np.array([2 if l == "D" else 1 for l in v.annot["lang"]])[None, :]
        ax.imshow(lang, aspect="auto", extent=(-0.5, v.n_tokens - 0.5, S.shape[0] - 0.5, S.shape[0] + 0.5),
                  cmap=cmap, vmin=0, vmax=2, interpolation="nearest")
        ax.set_ylim(-0.5, S.shape[0] + 0.5)
        ax.set_xlim(-0.5, v.n_tokens - 0.5)
        ax.axvline(v.tail_start - 0.5, color="k", lw=0.8, ls="--")
        ax.set_title(f"K={K} {Cc} ({v.tag}); oberste Zeile = Eingabesprache", fontsize=9)
        ax.set_ylabel("ℓ")
    axes[-1, 0].set_xlabel("Position t   (grau N, blau E, rot D)")
    savefig("ca_fields")
    plt.show()
    ca = table("D", "ca")
    if len(ca):
        cols = [c for c in ["tag", "K", "C", "spatial_entropy", "temporal_entropy", "activity", "lz76", "block_entropy_2x2", "frac_D", "frac_D_final_tail"] if c in ca.columns]
        show(ca[cols])


In [ ]:
# --- D5-D7: Wechselstatistik, Suszeptibilität χ(ε), Fluktuation-Antwort T_eff ----------
with guarded("Suszeptibilität und FDT"):
    su = table("D", "susceptibility")
    fdt = table("D", "fdt")
    sw = table("D", "switch_stats")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    for (tag, d, op), g in su.groupby(["tag", "direction", "order_parameter"]):
        if op != "fD":
            continue
        g = g.sort_values("eps")
        a1.plot(g["eps"], g["mean"], marker="o", ls="-" if d == "lang_dom" else "--", label=f"{tag} · {d}")
        a2.plot(g["eps"], g["chi"], marker="o", ls="-" if d == "lang_dom" else "--", label=f"{tag} · {d}")
    a1.set_xlabel("ε (konstant)")
    a1.set_ylabel("⟨f_D⟩ über Samples")
    a1.set_title("Ordnungsparameter f_D(ε)")
    a2.set_xlabel("ε")
    a2.set_ylabel("χ = d f_D / dε")
    a2.set_title("Suszeptibilität")
    a1.legend(fontsize=6)
    savefig("susceptibility")
    plt.show()
    if len(fdt):
        cols = [c for c in ["tag", "K", "C", "direction", "var0_fD", "chi_fD_1", "T_eff_fD_1", "var0_proj", "chi_proj_0", "T_eff_proj_0"] if c in fdt.columns]
        print("Fluktuation-Antwort (explorativ): T_eff = Var₀(m)/χ")
        show(fdt[cols])
    if len(sw):
        cols = [c for c in ["tag", "K", "direction", "source", "eps", "n_intervals", "geometric_p", "ks_stat", "tail_index"] if c in sw.columns]
        print("Wechselstatistik (geometrisch = Markov; schwere Ränder = Lawinen):")
        show(sw[cols], n=30)


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
